# DPR processor example with Prefect+Dask

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-520

See the associated Python module: [dpr_processor_example.py](./dpr_processor_example.py)

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/5341d2143d21486dabd08323ed8c9681/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
%%bash
prefect block ls
echo -e "\nNOTE: you can see the block details and credentials by running e.g.: prefect block inspect s3-bucket/s3"

                                  Blocks                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━┓
┃ ID                                   ┃ Type      ┃ Name ┃ Slug         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━┩
│ e371db8e-a34f-4162-94e6-deb972d3aa25 │ S3 Bucket │ s3   │ s3-bucket/s3 │
│ c2dea7c0-334a-4c47-a661-0a9bb6c42d4d │ Secret    │ auth │ secret/auth  │
└──────────────────────────────────────┴───────────┴──────┴──────────────┘
              List Block Types using `prefect block type ls`              

NOTE: you can see the block details and credentials by running e.g.: prefect block inspect s3-bucket/s3


In [4]:
# Other imports
import getpass
import os
from importlib import reload
from pathlib import Path
from resources import prefect_utils

# Test the DPR processing with n dummy products
output_count = 3
s3_basename = "new_zarr_product_"
s3_filenames = [f"{s3_basename}{i}" for i in range(output_count)]

# Data to test the example flow. 
# Depending on the functions we call, we need to pass a different dir (yikes)
s3_subdir = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/zarr"
s3_prefix_subdir = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_subdir}"
s3_full_path = f"s3://{PREFECT_BLOCK_S3.bucket_name}/{s3_prefix_subdir}"
my_data = {"s3_folder": s3_full_path, "s3_filenames": s3_filenames}

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

print(f"Output zarr products will be written to: {s3_full_path}")

Output zarr products will be written to: s3://prefect-share/sub/dir/users/jgaucher/zarr


In [5]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Set environment variables for the client and dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"
dask_client.run(set_dask_env)
os.environ["HELLO_FROM"] = "client"

In [6]:
# NOTE: we need to create the S3 folder with a dummy file before running DPR
empty_file = Path("/tmp/.empty")
empty_file.touch()
await PREFECT_BLOCK_S3.upload_from_path(empty_file, f"{s3_subdir}/{empty_file.name}")

15:11:01.540 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/.empty' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/zarr/.empty'.

'sub/dir/users/jgaucher/zarr/.empty'

# 1. Call Prefect flow from Python code
This is useful to debug or see the generated HTML representation, but in production we'll deploy the flow (see next section).

In [7]:
print(f"Remove existing zarr products from: {s3_full_path!r}")
PREFECT_BLOCK_S3._get_bucket_resource().objects.filter(Prefix=f"{s3_prefix_subdir}/{s3_basename}").delete()

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/zarr'


[]

In [8]:
# Import the module, or reload it if you changed its source code
import dpr_processor_example
reload(dpr_processor_example)

# Run the flow
results = dpr_processor_example.dpr_flow(**my_data)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | None   | 1.26.4    | 1.26.4  |
| pandas  | None   | 2.2.3     | 2.2.3   |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


15:11:01.989 | WARNING | root - Hello from 'client' '172.18.0.20' (main code)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | None   | 1.26.4    | 1.26.4  |
| pandas  | None   | 2.2.3     | 2.2.3   |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


15:11:02.093 | WARNING | root - Hello from 'client' '172.18.0.20' (main code)

15:11:02.173 | INFO    | prefect.engine - Created flow run 'quantum-magpie' for flow 'dpr-flow'

...

15:11:12.249 | ERROR   | Flow run 'quantum-magpie' - timed out after 10 s.

15:11:12.251 | INFO    | Flow run 'quantum-magpie' - Try #2

15:11:21.623 | INFO    | Flow run 'quantum-magpie' - Try #1

15:11:21.634 | INFO    | Flow run 'quantum-magpie' - Try #1

15:11:21.672 | INFO    | Flow run 'quantum-magpie' - Finished in state Completed()

In [9]:
# Display HTML representation
import IPython
for result in results:
    display(IPython.display.HTML(result))
del results

In [10]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

# Open them with the zarr python package
!pip install zarr
import zarr

total 20
drwxr-xr-x 5 jovyan users 4096 Feb  5 15:11 .
drwxrwxrwt 1 root   root  4096 Feb  5 15:11 ..
-rw-r--r-- 1 jovyan users    0 Feb  5 15:11 .empty
drwxr-xr-x 3 jovyan users 4096 Feb  5 15:11 new_zarr_product_0.zarr
drwxr-xr-x 3 jovyan users 4096 Feb  5 15:11 new_zarr_product_1.zarr
drwxr-xr-x 3 jovyan users 4096 Feb  5 15:11 new_zarr_product_2.zarr
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 39.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 38.8 MB/s eta 0:00:0000:010:01


In [11]:
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_0.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[78, 37, 34, ..., 31, 53, 68],
       [96, 12, 59, ..., 79, 57, 45],
       [90, 37, 37, ...,  4, 17, 60],
       ...,
       [ 5, 46, 82, ..., 14,  9, 18],
       [65, 48, 78, ..., 54, 66, 59],
       [92, 75, 75, ..., 33, 61, 39]], shape=(1024, 1024))

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_1.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[46, 27, 58, ..., 54, 95, 29],
       [76, 78, 10, ..., 39, 77, 79],
       [ 0,  9, 53, ...,  8, 16, 33],
       ...,
       [83, 90, 24, ..., 82, 72, 68],
       [29, 66, 10, ..., 88, 15, 82],
       [76,  0, 90, ..., 96, 14, 30]], shape=(1024, 1024))

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_2.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[95, 35, 33, ..., 22, 19, 89],
       [77, 81, 62, ..., 26, 90, 19],
       [81, 59, 82, ..., 67, 61, 71],
       ...,
       [95, 64, 49, ..., 92,  6,  1],
       [67, 31, 29, ...,  3, 97, 97],
       [17,  4,  3, ..., 50, 29, 44]], shape=(1024, 1024))

# 2. Deploy Prefect flow

See the full yaml file: [deploy-prefect-dask.yaml](./deploy-prefect-dask.yaml)

Deploy our source code via the S3 bucket.

NOTE: I have an error `Failed to parse headers (url=...dpr_processor_example.py)`, I don't understand why.

In [12]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


15:11:30.030 | WARNING | urllib3.connection - Failed to parse headers (url=http://minio:9000/prefect-share/sub/dir/users/jgaucher/code/dpr_processor_example.py): [MissingHeaderBodySeparatorDefect()], unparsed data: 'HTTP/1.1 200 OK\r\nAccept-Ranges: bytes\r\nContent-Length: 0\r\nETag: "c195175d89a59eb22cd3ce5f85808e87"\r\nServer: MinIO\r\nStrict-Transport-Security: max-age=31536000; includeSubDomains\r\nVary: Origin\r\nVary: Accept-Encoding\r\nX-Amz-Checksum-Crc32: GDAS1w==\r\nX-Amz-Id-2: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8\r\nX-Amz-Request-Id: 1821588C21D2699E\r\nX-Content-Type-Options: nosniff\r\nX-Ratelimit-Limit: 4782\r\nX-Ratelimit-Remaining: 4782\r\nX-Xss-Protection: 1; mode=block\r\nDate: Wed, 05 Feb 2025 15:11:30 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/urllib3/connection.py", line 464, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/opt/conda/lib/python3.11/site-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'HTTP/1.1 200 OK\r\nAccept-Ranges: bytes\r\nContent-Length: 0\r\nETag: "c195175d89a59eb22cd3ce5f85808e87"\r\nServer: MinIO\r\nStrict-Transport-Security: max-age=31536000; includeSubDomains\r\nVary: Origin\r\nVary: Accept-Encoding\r\nX-Amz-Checksum-Crc32: GDAS1w==\r\nX-Amz-Id-2: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8\r\nX-Amz-Request-Id: 1821588C21D2699E\r\nX-Content-Type-Options: nosniff\r\nX-Ratelimit-Limit: 4782\r\nX-Ratelimit-Remaining: 4782\r\nX-Xss-Protection: 1; mode=block\r\nDate: Wed, 05 Feb 2025 15:11:30 GMT\r\n\r\n'

In [13]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./dpr_processor_example.yaml"

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.2  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
15:12:02.738 | WARNING | root - Hello from 'client' '172.18.0.20' (main code)


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'dpr-flow/sprint20-dpr-example' successfully created with id      │
│ '725608ca-53c1-45cf-961e-ccb89d93f04d'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/725608ca-53c1-45cf-961e-ccb89d93f04d


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'dpr-flow/sprint20-dpr-example'



In [14]:
deploy_name = "dpr-flow/sprint20-dpr-example"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'dpr-flow/sprint20-dpr-example'


In [15]:
print(f"Remove existing zarr products from: {s3_full_path!r}")
PREFECT_BLOCK_S3._get_bucket_resource().objects.filter(Prefix=f"{s3_prefix_subdir}/{s3_basename}").delete()

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/zarr'


[{'ResponseMetadata': {'RequestId': '18215893E834D11A',
   'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'HTTPStatusCode': 200,
   'HTTPHeaders': {'accept-ranges': 'bytes',
    'content-length': '7163',
    'content-type': 'application/xml',
    'server': 'MinIO',
    'strict-transport-security': 'max-age=31536000; includeSubDomains',
    'vary': 'Origin, Accept-Encoding',
    'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
    'x-amz-request-id': '18215893E834D11A',
    'x-content-type-options': 'nosniff',
    'x-ratelimit-limit': '4782',
    'x-ratelimit-remaining': '4782',
    'x-xss-protection': '1; mode=block',
    'date': 'Wed, 05 Feb 2025 15:12:03 GMT'},
   'RetryAttempts': 0},
  'Deleted': [{'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zattrs'},
   {'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zgroup'},
   {'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zmetadata'}

In [16]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'dpr-flow/sprint20-dpr-example'...
Created flow run 'unnatural-mole'.
└── UUID: 37ba056f-a2a8-42cd-9417-93dcd5d28f99
└── Parameters: {'s3_folder': 's3://prefect-share/sub/dir/users/jgaucher/zarr', 's3_filenames': ['new_zarr_product_0', 'new_zarr_product_1', 'new_zarr_product_2']}
└── Job Variables: {}
└── Scheduled start time: 2025-02-05 15:12:04 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/37ba056f-a2a8-42cd-9417-93dcd5d28f99
Watching flow run 'unnatural-mole'...


15:12:04.679 | INFO    | prefect - Flow run is in state 'Scheduled'


15:12:08.891 | WARNING | Flow run 'unnatural-mole' - Hello from 'dask' '172.18.0.10' (new_zarr_product_0)

15:12:08.891 | WARNING | Flow run 'unnatural-mole' - Hello from 'dask' '172.18.0.10' (new_zarr_product_0)

15:12:08.891 | WARNING | Flow run 'unnatural-mole' - Hello from 'dask' '172.18.0.10' (new_zarr_product_0)

...

15:12:09.172 | ERROR   | Task run 'single_dpr_task' - Task run failed with exception: ContainsGroupError("path 'measurements' contains a group") - Retries are exhausted
Traceback (most recent call last):
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 781, in run_context
    yield self
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 1337, in run_task_sync
    engine.call_task_fn(txn)
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 804, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/utilities/callables.py", line 206, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/tmphkput0dsprefect/dpr_processor_example.py", line 301, in single_dpr_task
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/tasks.py", line 1002, in __call__
    return run_task(
           ^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 1526, in run_task
    return run_task_sync(**kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 1339, in run_task_sync
    return engine.state if return_type == "state" else engine.result()
                                                       ^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 482, in result
    raise self._raised
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 781, in run_context
    yield self
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 1337, in run_task_sync
    engine.call_task_fn(txn)
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/task_engine.py", line 804, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/prefect/utilities/callables.py", line 206, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/tmphkput0dsprefect/dpr_processor_example.py", line 275, in all_my_eopf_code
  File "/home/dask/.local/lib/python3.11/site-packages/eopf/store/zarr.py", line 684, in __setitem__
    self._write_eog(group_fspath, group_fspath.to_zarr_store(), __value)
  File "/home/dask/.local/lib/python3.11/site-packages/eopf/store/zarr.py", line 584, in _write_eog
    self._write_eog(
  File "/home/dask/.local/lib/python3.11/site-packages/eopf/store/zarr.py", line 557, in _write_eog
    delayed_zarr = ds.to_zarr(
                   ^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/xarray/core/dataset.py", line 2548, in to_zarr
    return to_zarr(  # type: ignore[call-overload,misc]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/xarray/backends/api.py", line 1661, in to_zarr
    zstore = backends.ZarrStore.open_group(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/xarray/backends/zarr.py", line 500, in open_group
    zarr_group = zarr.open_group(store, **open_kwargs)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dask/.local/lib/python3.11/site-packages/zarr/hierarchy.py", line 1593, in open_group
    raise ContainsGroupError(path)
zarr.errors.ContainsGroupError: path 'measurements' contains a group

15:12:09.183 | ERROR   | Task run 'single_dpr_task' - Finished in state Failed("Task run encountered an exception ContainsGroupError: path 'measurements' contains a group")

15:12:09.698 | INFO    | prefect - Flow run is in state 'Failed'


Flow run finished in state 'Failed'.


CalledProcessError: Command 'b'# Trigger a run for this flow from the command line\nprefect deployment run "$1" --params "$2" --watch\n'' returned non-zero exit status 1.

In [ ]:
# Download zarr products into local (same as above)
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

In [ ]:
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

## 3. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

15:13:41.593 | WARNING | root - Hello from 'dask' '172.18.0.10' (eopf)

15:13:41.593 | WARNING | root - Hello from 'dask' '172.18.0.10' (eopf)

15:13:41.593 | WARNING | root - Hello from 'dask' '172.18.0.10' (eopf)

15:13:41.609 | WARNING | root - Hello from 'dask' '172.18.0.10' (eopf)

15:13:41.609 | WARNING | root - Hello from 'dask' '172.18.0.10' (eopf)

...

15:13:45.547 | WARNING | root -  emoji methods/attributes: ['EMOJI_DATA', 'EmojiMatch', 'EmojiMatchZWJ', 'EmojiMatchZWJNonRGI', 'LANGUAGES', 'STATUS', 'Token', '__all__', '__author__', '__builtins__', '__cached__', '__doc__', '__email__', '__file__', '__license__', '__loader__', '__name__', '__package__', '__path__', '__source__', '__spec__', '__version__', 'analyze', 'config', 'core', 'demojize', 'distinct_emoji_list', 'emoji_count', 'emoji_list', 'emojize', 'get_emoji_by_name', 'is_emoji', 'load_from_json', 'purely_emoji', 'replace_emoji', 'tokenizer', 'unicode_codes', 'version']

15:13:45.547 | WARNING | root -  emoji methods/attributes: ['EMOJI_DATA', 'EmojiMatch', 'EmojiMatchZWJ', 'EmojiMatchZWJNonRGI', 'LANGUAGES', 'STATUS', 'Token', '__all__', '__author__', '__builtins__', '__cached__', '__doc__', '__email__', '__file__', '__license__', '__loader__', '__name__', '__package__', '__path__', '__source__', '__spec__', '__version__', 'analyze', 'config', 'core', 'demojize', 'distinct_emoji_list', 'emoji_count', 'emoji_list', 'emojize', 'get_emoji_by_name', 'is_emoji', 'load_from_json', 'purely_emoji', 'replace_emoji', 'tokenizer', 'unicode_codes', 'version']

15:14:18.831 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

15:14:18.832 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

15:14:18.832 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client